# Exemplo 15 — Detectar ausência e duplicação por contador

Envia uma sequência pelo VirtualBus, retirando dois contadores e duplicando um para criar um caso controlado. Calcula IDs ausentes e duplicados após a decodificação.

No Colab, execute as células de cima para baixo e envie o ZIP `Protocolos_Veiculares_36_Exemplos_Separados.zip` quando solicitado. A primeira célula extrai somente a pasta deste exercício. Não precisa executar outro exemplo nem fornecer imagem ou vídeo.

No VS Code, abra a pasta deste exemplo e prefira o arquivo `.py`. O notebook também reconhece a pasta local quando ela é o diretório de trabalho.

Observe: As ausências são introduzidas no publicador para estudo; não são erros de bit medidos em CAN físico.

In [ ]:
# Localizar o exercício ou extrair apenas sua pasta do ZIP enviado ao Colab.
from pathlib import Path, PurePosixPath
import io, sys, zipfile, subprocess
NOME_EXEMPLO = '15_contador_e_ausencias'
ARQUIVO_PY = '15_contador_e_ausencias.py'
try:
    from google.colab import files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
PASTA_EXEMPLO = Path.cwd()
if not (PASTA_EXEMPLO / ARQUIVO_PY).is_file():
    if not EM_COLAB:
        raise RuntimeError('Abra a pasta deste exemplo como diretório de trabalho no VS Code.')
    enviados = files.upload()
    arquivos_zip = [(n, b) for n, b in enviados.items() if n.lower().endswith('.zip')]
    if not arquivos_zip:
        raise ValueError('Envie o ZIP dos 36 exemplos separados.')
    PASTA_EXEMPLO = Path('/content/protocolos_separados') / NOME_EXEMPLO
    PASTA_EXEMPLO.mkdir(parents=True, exist_ok=True)
    extraidos = 0
    with zipfile.ZipFile(io.BytesIO(arquivos_zip[0][1])) as z:
        for info in z.infolist():
            partes = PurePosixPath(info.filename).parts
            if NOME_EXEMPLO not in partes or info.is_dir():
                continue
            relativo = PurePosixPath(*partes[partes.index(NOME_EXEMPLO) + 1:])
            if not relativo.parts or relativo.is_absolute() or '..' in relativo.parts:
                raise ValueError('Caminho inválido no ZIP.')
            alvo = PASTA_EXEMPLO.joinpath(*relativo.parts)
            alvo.parent.mkdir(parents=True, exist_ok=True)
            alvo.write_bytes(z.read(info))
            extraidos += 1
    if not (PASTA_EXEMPLO / ARQUIVO_PY).is_file():
        raise ValueError('O ZIP enviado não contém a pasta deste exemplo. Use a versão com exemplos separados.')
    print('Arquivos extraídos:', extraidos)
print('Pasta do exercício:', PASTA_EXEMPLO)

In [ ]:
# Instalar as bibliotecas: python-can, OpenCV, NumPy e Matplotlib.
# A versão headless do OpenCV evita janelas nativas no Colab.
requisitos = 'requirements-colab.txt' if EM_COLAB else 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(PASTA_EXEMPLO / requisitos)], check=True)

## Executar o exemplo

O comando abaixo inicia o `.py` em um processo Python separado. Isso mantém o mesmo funcionamento no notebook e no terminal, inclusive nos exemplos com publicador, assinante, servidor e cliente. As medições são feitas no computador que executa o código.

Os gráficos e feeds são salvos em `resultados/`. Os exemplos 28 e 36 transmitem 320 frames por protocolo e podem levar cerca de 25 segundos, ou mais conforme a máquina.

In [ ]:
# Executar somente este exercício; não depende de resultados anteriores.
comando = [sys.executable, str(PASTA_EXEMPLO / ARQUIVO_PY), '--sem-gui']
subprocess.run(comando, cwd=PASTA_EXEMPLO, check=True)
PASTA_RESULTADOS = PASTA_EXEMPLO / 'resultados' / Path(ARQUIVO_PY).stem
print('Resultados:', PASTA_RESULTADOS)

In [ ]:
# Mostrar figuras, relatórios e, quando existir, um feed recebido.
from IPython.display import display, Image, Markdown, Video
import shutil
for figura in sorted(PASTA_RESULTADOS.rglob('*.png'))[:12]:
    print(figura.relative_to(PASTA_RESULTADOS))
    display(Image(filename=str(figura)))
for relatorio in sorted(PASTA_RESULTADOS.rglob('relatorio*.md'))[:2]:
    display(Markdown(relatorio.read_text(encoding='utf-8')))
for feed in sorted(PASTA_RESULTADOS.rglob('feed_recebido.mp4'))[:2]:
    if shutil.which('ffmpeg'):
        navegavel = feed.with_name('feed_para_notebook.mp4')
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(feed),
                        '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                        '-movflags', '+faststart', str(navegavel)], check=True)
        display(Video(str(navegavel), embed=True, width=640))
    else:
        print('Feed salvo:', feed, '— use VLC ou instale ffmpeg para exibir aqui.')
if not list(PASTA_RESULTADOS.rglob('*.png')):
    print('Este exercício apresenta seus resultados no terminal e em arquivos.')